# Modern AABB quatrain generation (L40 + vLLM)

One notebook for both the test and the real run. There are two switches in the settings cell:

- **`MODE`**
  - `"pilot"`: about 300 prompts, a few minutes. Shows pass rates, judge scores, samples and a time estimate. Output: `pilot_v3_out/<model>/`.
  - `"full"`: generates in chunks until `TARGET_USABLE` quatrains are saved to `gen_out/<model>/quatrains.jsonl`. Saved after every chunk; rerun the notebook to resume after a crash.
- **`MODEL`**: which model to use, picked by name from the presets (`qwen3-30b-a3b`, `qwen3-32b`, `gemma3-27b`, `qwen2.5-32b`). **Restart the kernel after changing it**, because vLLM keeps the previous model on the GPU.

Changes from v2:
1. **End-word tags.** Each line is written as `[air] The scent lingers soft in the car's air,` so the target word sits right before the line. In v2 the model ended a line with its planned word only about 40% of the time.
2. **Quality judge.** Candidates that pass the form checks are rated 1–5 for sense and naturalness by the same model, and anything below `JUDGE_MIN` is dropped. The form filters cannot catch lines like *"a memory spinning without a round"*.
3. **Fixed rhyme rule.** v2 rejected real rhymes such as air/chair, one/done and in/grin. Now a pair is only rejected when one word is the other with extra syllables in front (play/replay, side/inside) or they sound the same (one/won).
4. **Model presets** and a **comparison table** at the end that reads every pilot you have run.

**Workflow:** run the pilot for 2–3 models (change `MODEL`, restart, Run All) → compare in the last cell and read the samples → set `MODE = "full"` with the winner → Restart & Run All.

## 1. Install

In [1]:
%pip install -q vllm pronouncing wordfreq pandas

Note: you may need to restart the kernel to use updated packages.


## 2. Settings

**`MODE` and `MODEL` are the switches.** Everything else can stay at its default.

In [2]:
import os, re, json, time, random
from collections import Counter, defaultdict

# ======================= THE SWITCHES =======================
MODE  = "full"             # "pilot" = small test with samples | "full" = real run, chunked and resumable
MODEL = "qwen3-32b"     # a key from MODELS below; RESTART THE KERNEL after changing it
# ============================================================

MODELS = {
    # mixture-of-experts, 3B active per token: very fast, weaker at sense (the v2 baseline)
    "qwen3-30b-a3b": dict(id="Qwen/Qwen3-30B-A3B-Instruct-2507-FP8"),
    # dense 32B in FP8, thinking switched off
    "qwen3-32b":     dict(id="Qwen/Qwen3-32B-FP8",
                          chat_kwargs={"chat_template_kwargs": {"enable_thinking": False}}),
    # dense 27B, quantised to FP8 when loaded; needs `huggingface-cli login` and the licence accepted on its page
    "gemma3-27b":    dict(id="google/gemma-3-27b-it",
                          llm_kwargs={"quantization": "fp8", "limit_mm_per_prompt": {"image": 0}}),
    # dense 32B, 4-bit AWQ
    "qwen2.5-32b":   dict(id="Qwen/Qwen2.5-32B-Instruct-AWQ"),
}
assert MODE in ("pilot", "full") and MODEL in MODELS
MODEL_ID = MODELS[MODEL]["id"]

# --- run size ---
PILOT_PROMPTS = 300         # pilot: number of prompts
TARGET_USABLE = 20_000      # full: stop once this many quatrains are saved (try 50_000 later)
CHUNK_PROMPTS = 2_000       # full: prompts per chunk; results are saved after each chunk
MAX_CHUNKS    = 200         # full: safety stop

# --- sampling ---
N_CANDIDATES = 4            # samples per prompt; the best passing one is kept
TEMPERATURE  = 0.9
TOP_P        = 0.95
MAX_TOKENS   = 200          # RHYMES line + 4 tagged lines is about 90-110 tokens
SEED         = 42

# --- quality judge ---
JUDGE     = True            # rate candidates that pass the form checks for sense and naturalness
JUDGE_MIN = 4               # keep scores >= this (1-5 scale)

# --- form filters ---
SYL_MIN, SYL_MAX = 6, 12    # allowed syllables per line
PAIR_SYL_DIFF    = 3        # max syllable difference within a couplet
MAX_OOV_RATE     = 0.15     # max share of words not in CMUdict
MIN_END_ZIPF     = 2.5      # end words must be at least this common (wordfreq Zipf scale); blocks names and made-up words
MAX_PAIR_USES    = 50       # max times one rhyme pair (e.g. name/game) may appear in the dataset
NEAR_DUP_ROUGE   = 0.7      # ROUGE-L F1 above this = near-duplicate

PILOT_ROOT  = "pilot_v3_out"
OUT_DIR     = f"gen_out/{MODEL}" if MODE == "full" else f"{PILOT_ROOT}/{MODEL}"
TOPICS_FILE = "topics.json" # shared by all models and modes
os.makedirs(OUT_DIR, exist_ok=True)
print("MODE:", MODE, "| MODEL:", MODEL, f"({MODEL_ID})", "| output:", OUT_DIR)

MODE: pilot | MODEL: qwen2.5-32b (Qwen/Qwen2.5-32B-Instruct-AWQ) | output: pilot_v3_out/qwen2.5-32b


## 3. Phonetic helpers

In [3]:
import pronouncing
from wordfreq import zipf_frequency

WORD_RE   = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")
QUOTE_MAP = str.maketrans({"’": "'", "‘": "'", "“": '"', "”": '"'})

def normalize(text):
    return text.translate(QUOTE_MAP)        # curly quotes -> straight, so "I’ll" stays one word

def words_of(line):
    return [w.lower() for w in WORD_RE.findall(normalize(line))]

def last_word(line):
    w = words_of(line)
    return w[-1] if w else ""

def all_phones(word):
    return pronouncing.phones_for_word(word.lower())

def rhyme_parts(word):
    return {pronouncing.rhyming_part(p) for p in all_phones(word)}

def same_sound(a, b):
    # True if a and b share a rhyme sound (identical words count)
    ra, rb = rhyme_parts(a), rhyme_parts(b)
    return bool(ra and rb and ra & rb)

def identical_rhyme(pa, pb):
    # Not a real rhyme if one pronunciation is the other plus extra syllables in front
    # (play/replay, side/inside, way/away) or they sound the same (one/won).
    # Extra consonants are fine: air/chair, red/spread, in/grin.
    a, b = pa.split(), pb.split()
    short, long_ = sorted((a, b), key=len)
    if long_[-len(short):] != short:
        return False
    extra = long_[:len(long_) - len(short)]
    return not extra or any(p[-1].isdigit() for p in extra)

def rhymes(a, b):
    a, b = a.lower(), b.lower()
    if not a or not b or a == b:
        return False
    return any(pronouncing.rhyming_part(pa) == pronouncing.rhyming_part(pb) and not identical_rhyme(pa, pb)
               for pa in all_phones(a) for pb in all_phones(b))

def word_syllables(word):
    ph = all_phones(word)
    if ph:
        return pronouncing.syllable_count(ph[0])
    groups = re.findall(r"[aeiouy]+", word.lower())       # fallback for unknown words
    n = len(groups)
    if word.lower().endswith("e") and n > 1 and not word.lower().endswith("le"):
        n -= 1
    return max(n, 1)

def line_syllables(line):
    return sum(word_syllables(w) for w in words_of(line))

def oov_rate(lines):
    ws = [w for l in lines for w in words_of(l)]
    return sum(1 for w in ws if not all_phones(w)) / len(ws) if ws else 1.0

for a, b in [("late", "wait"), ("phone", "alone"), ("air", "chair"), ("one", "done"), ("in", "grin"),
             ("red", "spread"), ("trace", "grace"), ("up", "cup")]:
    assert rhymes(a, b), (a, b)
for a, b in [("late", "phone"), ("play", "replay"), ("side", "inside"), ("way", "away"), ("one", "won")]:
    assert not rhymes(a, b), (a, b)
assert words_of("I’ll go") == ["i'll", "go"]
print("helpers ok")

helpers ok


## 4. Load the model

If this fails for `gemma3-27b`, log in first (`huggingface-cli login`) and accept the licence on the model page. If a model runs out of memory, lower `max_model_len` to 3072 or `gpu_memory_utilization` to 0.85.

In [4]:
from vllm import LLM, SamplingParams

M = MODELS[MODEL]
llm = LLM(
    model=MODEL_ID,
    max_model_len=4096,
    gpu_memory_utilization=0.90,
    enable_prefix_caching=True,   # the system prompt and examples are shared, so they are cached
    seed=SEED,
    **M.get("llm_kwargs", {}),
)
CHAT_KW = M.get("chat_kwargs", {})
params = SamplingParams(n=N_CANDIDATES, temperature=TEMPERATURE, top_p=TOP_P,
                        max_tokens=MAX_TOKENS, repetition_penalty=1.05)
judge_params = SamplingParams(temperature=0.0, max_tokens=4)

def chat(msgs, sp, progress=False):
    return llm.chat(msgs, sp, use_tqdm=progress, **CHAT_KW)

/common/home/projectgrps/CS480/CS480G3/jupyterlab-venv-pytorch-240/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(MainProcess pid=3251142) INFO 10-20 10:36:07 [api_utils.py:286] non-default args: {'seed': 42, 'max_model_len': 4096, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-32B-Instruct-AWQ'}


(MainProcess pid=3251142) INFO 10-20 10:36:16 [model.py:733] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=3251142) INFO 10-20 10:36:16 [model.py:2152] Using max model len 4096
(MainProcess pid=3251142) INFO 10-20 10:36:17 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.


Parse safetensors files: 100%|██████████| 5/5 [00:01<00:00,  4.47it/s]

(MainProcess pid=3251142) INFO 10-20 10:36:18 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


...

(EngineCore pid=3251245) INFO 10-20 10:39:17 [default_loader.py:501] Loading weights took 5.18 seconds
(EngineCore pid=3251245) INFO 10-20 10:39:21 [model_runner.py:424] Model loading took 18.14 GiB memory and 179.395074 seconds
(EngineCore pid=3251245) INFO 10-20 10:39:21 [topk_topp_sampler.py:124] Using FlashInfer for top-p & top-k sampling.
(EngineCore pid=3251245) INFO 10-20 10:39:21 [torch_utils.py:295] Reducing Torch threads from 4 to 1 for serving. Set OMP_NUM_THREADS in the external environment to override.
(EngineCore pid=3251245) INFO 10-20 10:39:21 [utils.py:347] Using LBNHC KV cache layout.
(EngineCore pid=3251245) INFO 10-20 10:39:21 [monitor.py:40] [backbone] Starting torch.compile
(EngineCore pid=3251245) INFO 10-20 10:39:31 [backends.py:1104] [backbone] Using cache directory: /common/home/projectgrps/CS480/CS480G3/.cache/vllm/torch_compile_cache/e56b750ffd/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=3251245) INFO 10-20 10:39:31 [backends.py:1166] [backbon

Capturing CUDA graphs (FULL): 100%|██████████| 2/2 [00:00<00:00,  8.96it/s]


(EngineCore pid=3251245) INFO 10-20 10:39:59 [model_runner.py:1127] Graph capturing finished in 8 secs, took 0.71 GiB
(EngineCore pid=3251245) INFO 10-20 10:39:59 [gpu_worker.py:759] Available KV cache memory: 19.12 GiB
(EngineCore pid=3251245) INFO 10-20 10:39:59 [gpu_worker.py:774] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9000 is equivalent to --gpu-memory-utilization=0.8824 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9176. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=3251245) INFO 10-20 10:39:59 [kv_cache_utils.py:2327] GPU KV cache size: 78,304 tokens, Maximum concurrency for 4,096 tokens per request: 19.12x
(EngineCore pid=3251245) INFO 10-20 10:39:59 [kernel_warmup.py:176] JIT kernel warmup starting.
(EngineCore pid=3251245) INFO 10-20 10:39:59 [kernel_warmup.py:189] JIT kernel warmup finished in 0.00s

Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:02<00:00, 13.45it/s]


(EngineCore pid=3251245) INFO 10-20 10:40:13 [model_runner.py:1127] Graph capturing finished in 10 secs, took 0.39 GiB
(EngineCore pid=3251245) INFO 10-20 10:40:13 [gpu_worker.py:971] CUDA graph pool memory: 0.39 GiB (actual), 0.78 GiB (estimated), difference: 0.39 GiB (98.5%).
(EngineCore pid=3251245) INFO 10-20 10:40:13 [gpu_worker.py:1034] Free memory on device (43.97/44.39 GiB) on startup. Desired GPU memory utilization is (0.9, 39.95 GiB). Actual usage is 18.46 GiB for consumed memory (weights + non-torch), 2.38 GiB for peak activation, and 0.39 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=19951917261` (18.58 GiB) to fit into requested memory, or `--kv-cache-memory=24263347200` (22.6 GiB) to fully utilize gpu memory. Current kv cache memory in use is 19.12 GiB.
(EngineCore pid=3251245) INFO 10-20 10:40:13 [jit_monitor.py:84] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=32512

## 5. Topics

About 600 everyday topics: the model writes 30 per category once, and they are cached in `topics.json` (shared by all models, so the comparison is fair). Delete that file to regenerate, or edit it by hand to remove weak topics.

In [5]:
BASE_TOPICS = [
    "missing the morning bus", "a group project where nobody helps", "the last slice of pizza",
    "a phone at one percent battery", "Monday mornings", "a cat that ignores you", "rainy weekends",
    "a gym membership you never use", "waiting for exam results", "a noisy neighbour",
    "online shopping regrets", "a houseplant that keeps dying", "the first day at a new job",
    "a friend who is always late", "burnt toast", "a long queue at the coffee shop",
    "forgetting someone's name", "the summer heat", "a road trip with bad directions",
    "a video call with a frozen screen", "a messy bedroom", "learning to cook", "a broken umbrella",
    "staying up too late", "a lost set of keys", "a birthday nobody remembered", "a cheap haircut",
    "sending a text to the wrong person", "the alarm clock", "a crowded train", "homework due tomorrow",
    "a dog that steals socks", "a failed diet", "an awkward first date", "the office printer",
    "a slow wifi connection", "grocery shopping while hungry", "a song stuck in your head",
    "moving to a new city", "a rainy football match", "a talkative taxi driver", "a group chat",
    "trying to wake up early", "a surprise party", "a spilled drink", "the weekend ending too soon",
    "a long flight", "a boring meeting", "a forgotten password", "the smell of fresh bread",
]
TOPIC_CATEGORIES = [
    "work and the office", "school and university", "cooking and food", "phones, apps and the internet",
    "pets", "weather and seasons", "public transport and driving", "family life", "friendship",
    "dating and relationships", "sport and the gym", "health and sleep", "money and bills",
    "home, chores and neighbours", "shopping", "holidays and travel", "social media",
    "birthdays and celebrations", "music and films", "hobbies", "city life", "nature and the outdoors",
    "childhood memories", "getting older", "small daily annoyances",
]

def generate_topics():
    msgs = [[{"role": "user", "content":
              f"List 30 specific everyday situations about {cat} that a short, light modern poem could be about. "
              "Each one is 3 to 8 words and concrete, like 'a cat that ignores you' or 'missing the morning bus'. "
              "Return only a JSON array of strings."}] for cat in TOPIC_CATEGORIES]
    outs = chat(msgs, SamplingParams(temperature=0.8, top_p=0.95, max_tokens=900))
    found = []
    for o in outs:
        text = o.outputs[0].text
        m = re.search(r"\[.*\]", text, re.S)
        try:
            found += [t for t in json.loads(m.group(0)) if isinstance(t, str)]
        except Exception:                     # fall back to one topic per line
            found += [re.sub(r'^[\s\-*\d.)"]+|[",]+$', "", l) for l in text.splitlines()]
    return found

if os.path.exists(TOPICS_FILE):
    TOPICS = json.load(open(TOPICS_FILE, encoding="utf-8"))
else:
    seen, TOPICS = set(), []
    for t in BASE_TOPICS + generate_topics():
        t = normalize(t).strip().rstrip(".")
        if 2 <= len(t.split()) <= 10 and t.lower() not in seen:
            seen.add(t.lower()); TOPICS.append(t)
    json.dump(TOPICS, open(TOPICS_FILE, "w", encoding="utf-8"), indent=1)

TONES = ["funny", "sarcastic", "wistful", "hopeful", "deadpan", "dramatic", "playful", "grumpy",
         "tender", "nostalgic", "anxious", "cheerful", "bittersweet", "proud"]
VOICES = ["first person (I)", "second person (you)", "a narrator describing someone"]
print(f"{len(TOPICS)} topics, e.g.", random.Random(0).sample(TOPICS, 6))

797 topics, e.g. ['itchy eyes from screens', 'glue stick dried out', 'receipts scattered on the table', 'a group chat', 'kitchen light left on', 'restaurant with no menu in English']


## 6. Prompt

Two of the four worked examples are given as earlier chat turns, chosen at random for each prompt. The model writes a `RHYMES:` plan line, then each line with its end word in square brackets at the front. The parser removes the plan line and the tags.

In [6]:
SYSTEM = (
    "You write short modern poems in rhyming couplets. Reply with a RHYMES line, then exactly 4 lines "
    "of poetry, each starting with its end word in square brackets. No title, no blank lines, no explanation."
)

EXAMPLES = [
    dict(topic="a phone at one percent battery", tone="dramatic", voice="first person (I)",
         rhymes="red / bed ; in / win",
         lines=["My battery icon has just turned red,",
                "I search for a charger under the bed.",
                "I find one at last and I plug it right in,",
                "the cable is broken, I just can't win."]),
    dict(topic="the office printer", tone="grumpy", voice="a narrator describing someone",
         rhymes="nine / line ; tray / day",
         lines=["He needs ten copies by a quarter past nine,",
                "the printer just blinks at the end of the line.",
                "He opens each drawer and he empties each tray,",
                "then writes them by hand for the rest of the day."]),
    dict(topic="rainy weekends", tone="wistful", voice="second person (you)",
         rhymes="sun / one ; street / sweet",
         lines=["You had big plans for the park and the sun,",
                "the rain cancelled each of them one by one.",
                "So you make some tea and you watch the street,",
                "and find that a slow day is almost as sweet."]),
    dict(topic="a group chat", tone="sarcastic", voice="first person (I)",
         rhymes="night / right ; all / small",
         lines=["The group chat pings a hundred times at night,",
                "four people arguing about who's right.",
                "I scroll back up to see what started it all:",
                "a photo of a sandwich, slightly small."]),
]

def user_prompt(topic, tone, voice):
    return (
        "Write one quatrain made of two rhyming couplets (AABB).\n"
        "- Line 1 rhymes with line 2. Line 3 rhymes with line 4.\n"
        "- The second couplet uses a different rhyme sound. Line 1 must not rhyme with line 3.\n"
        "- 7 to 11 syllables per line.\n"
        "- Every line must make sense on its own. Never add words just to reach a rhyme.\n"
        "- Modern everyday English. No archaic words, no cliches, simple punctuation.\n"
        "- Avoid overused rhymes like night/light, heart/apart, day/way.\n"
        f"Topic: {topic}\nTone: {tone}\nVoice: {voice}\n"
        "First write the RHYMES line with the end words of each couplet. Then write the 4 lines; "
        "start each one with its end word in square brackets, and end the line with exactly that word."
    )

def example_answer(ex):
    ends = re.findall(r"[a-z']+", ex["rhymes"])
    return "RHYMES: " + ex["rhymes"] + "\n" + "\n".join(f"[{e}] {l}" for e, l in zip(ends, ex["lines"]))

def build_messages(topic, tone, voice, rng):
    msgs = [{"role": "system", "content": SYSTEM}]
    for ex in rng.sample(EXAMPLES, 2):
        msgs.append({"role": "user", "content": user_prompt(ex["topic"], ex["tone"], ex["voice"])})
        msgs.append({"role": "assistant", "content": example_answer(ex)})
    msgs.append({"role": "user", "content": user_prompt(topic, tone, voice)})
    return msgs

def make_jobs(n, rng, start_id):
    jobs = []
    for i in range(n):
        topic, tone, voice = rng.choice(TOPICS), rng.choice(TONES), rng.choice(VOICES)
        jobs.append({"id": start_id + i, "topic": topic, "tone": tone, "voice": voice,
                     "messages": build_messages(topic, tone, voice, rng)})
    return jobs

print(user_prompt("a lost set of keys", "funny", VOICES[0]), "\n")
print(example_answer(EXAMPLES[0]))

Write one quatrain made of two rhyming couplets (AABB).
- Line 1 rhymes with line 2. Line 3 rhymes with line 4.
- The second couplet uses a different rhyme sound. Line 1 must not rhyme with line 3.
- 7 to 11 syllables per line.
- Every line must make sense on its own. Never add words just to reach a rhyme.
- Modern everyday English. No archaic words, no cliches, simple punctuation.
- Avoid overused rhymes like night/light, heart/apart, day/way.
Topic: a lost set of keys
Tone: funny
Voice: first person (I)
First write the RHYMES line with the end words of each couplet. Then write the 4 lines; start each one with its end word in square brackets, and end the line with exactly that word. 

RHYMES: red / bed ; in / win
[red] My battery icon has just turned red,
[bed] I search for a charger under the bed.
[in] I find one at last and I plug it right in,
[win] the cable is broken, I just can't win.


## 7. Checks, judge and dataset filters

**Per candidate (form):** 4 lines, AABB rhyme (ABAB is reported separately), different A and B sounds, syllable range, balanced couplets, unknown-word rate, rare or made-up end words.

**Judge:** candidates that pass the form checks are rated 1–5 for sense and naturalness. The rubric includes four calibration examples taken from the v2 pilot.

**Against the dataset so far:** exact duplicate, rhyme-pair cap, reused lines, near-duplicate (ROUGE-L, compared only with quatrains sharing a rhyme pair, so it stays fast at 50k).

In [7]:
THINK_RE = re.compile(r"<think>.*?</think>", re.S)
TAG_RE   = re.compile(r"^\[([^\]]*)\]\s*")

def parse(raw):
    text = normalize(THINK_RE.sub("", raw))
    plan, lines, tags = None, [], []
    for l in text.strip().splitlines():
        l = l.strip()
        if len(l) > 1 and l[0] == l[-1] == '"':
            l = l[1:-1].strip()
        if not l:
            continue
        if l.upper().startswith("RHYMES"):
            plan = l.split(":", 1)[-1].strip()
            continue
        l = re.sub(r"^(\d+[\.\):]|[-*•])\s*", "", l)       # drop numbering / bullets
        m = TAG_RE.match(l)
        tag = m.group(1).strip().lower() if m else None
        if m:
            l = l[m.end():].strip()
        if not l or l.lower().startswith(("here", "title", "sure")):
            continue
        lines.append(l); tags.append(tag)
    return plan, lines, tags

def check(lines):
    r = {"lines": lines, "ends": None, "syllables": None, "oov": None, "reasons": []}
    if len(lines) != 4:
        r["reasons"].append(f"line_count={len(lines)}")
    else:
        ends = [last_word(l) for l in lines]
        r["ends"] = ends
        a_ok, b_ok = rhymes(ends[0], ends[1]), rhymes(ends[2], ends[3])
        if not a_ok and not b_ok and same_sound(ends[0], ends[2]) and same_sound(ends[1], ends[3]):
            r["reasons"].append("ABAB")
        else:
            if not a_ok: r["reasons"].append("A_no_rhyme")
            if not b_ok: r["reasons"].append("B_no_rhyme")
            if a_ok and b_ok and same_sound(ends[0], ends[2]): r["reasons"].append("same_sound_AB")
        syl = [line_syllables(l) for l in lines]
        r["syllables"] = syl
        if any(s < SYL_MIN or s > SYL_MAX for s in syl):
            r["reasons"].append("syllable_range")
        if abs(syl[0] - syl[1]) > PAIR_SYL_DIFF or abs(syl[2] - syl[3]) > PAIR_SYL_DIFF:
            r["reasons"].append("pair_unbalanced")
        if any(zipf_frequency(e, "en") < MIN_END_ZIPF for e in ends):
            r["reasons"].append("rare_end_word")
        r["oov"] = round(oov_rate(lines), 3)
        if r["oov"] > MAX_OOV_RATE:
            r["reasons"].append("oov")
    r["form_pass"] = r["pass"] = not r["reasons"]
    return r

JUDGE_PROMPT = """Rate this short modern poem from 1 to 5 for sense and naturalness. Ignore rhyme and metre, which are checked separately.
5 = every line makes sense, sounds like something a person would really say, and fits the topic
4 = good overall, at most one slightly awkward phrase
3 = understandable, but at least one phrase is there only for the rhyme or does not quite make sense
2 = several forced or nonsensical phrases
1 = mostly nonsense or off topic

Calibration examples:
"She hums that tune again, soft as a play, / like it's still the morning, not quite today. / It drifts through the house, a quiet sound, / a memory spinning without a round." -> 2
"I run out the door before the morning rush, / just need one latte to bring her back to hush. / She'll take it slow and give me a real smile, / and maybe we'll talk for a while." -> 3
"You smile, but I can't say your name, / it's vanished like a lost game. / I fumble through my mental file, / then just call you 'buddy' for a while." -> 4
"You had big plans for the park and the sun, / the rain cancelled each of them one by one. / So you make some tea and you watch the street, / and find that a slow day is almost as sweet." -> 5

Topic: {topic}
Poem:
{poem}

Reply with only the number."""

def judge(items):
    # items: [(topic, lines)] -> scores 1-5 (0 if the reply could not be read)
    if not items:
        return []
    outs = chat([[{"role": "user", "content": JUDGE_PROMPT.format(topic=t, poem="\n".join(ls))}]
                 for t, ls in items], judge_params)
    return [int(m.group(0)) if (m := re.search(r"[1-5]", o.outputs[0].text)) else 0 for o in outs]

def near_dup(a, ca, b, cb):
    # ROUGE-L F1 > threshold? Word overlap is an upper bound on the LCS, so most pairs skip the DP.
    f1 = lambda n: 0.0 if n == 0 else 2 * n / (len(a) + len(b))
    if f1(sum((ca & cb).values())) <= NEAR_DUP_ROUGE:
        return False
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return f1(prev[-1]) > NEAR_DUP_ROUGE

class Corpus:
    """Accepted quatrains plus the indexes used for duplicate and rhyme-pair checks."""
    def __init__(self):
        self.items, self.keys = [], set()
        self.pair_uses, self.line_seen = Counter(), Counter()
        self.by_pair = defaultdict(list)          # rhyme pair -> (tokens, Counter) of accepted quatrains

    @staticmethod
    def pairs(ends):
        return [tuple(sorted(ends[0:2])), tuple(sorted(ends[2:4]))]

    def reject_reason(self, lines, ends):
        if " / ".join(l.lower() for l in lines) in self.keys:
            return "duplicate"
        ps = self.pairs(ends)
        if any(self.pair_uses[p] >= MAX_PAIR_USES for p in ps):
            return "pair_cap"
        if sum(self.line_seen[l.lower()] > 0 for l in lines) >= 2:
            return "reused_lines"
        toks = [w for l in lines for w in words_of(l)]
        c = Counter(toks)
        if any(near_dup(toks, c, t, ct) for p in ps for t, ct in self.by_pair[p]):
            return "near_duplicate"
        return None

    def add(self, rec, store=True):
        lines, ends = rec["lines"], [last_word(l) for l in rec["lines"]]
        self.keys.add(" / ".join(l.lower() for l in lines))
        toks = [w for l in lines for w in words_of(l)]
        for p in self.pairs(ends):
            self.pair_uses[p] += 1
            self.by_pair[p].append((toks, Counter(toks)))
        for l in lines:
            self.line_seen[l.lower()] += 1
        if store:
            self.items.append(rec)

def pick(cands, corpus):
    ok = []
    for c in cands:
        if not c["pass"]:
            continue
        why = corpus.reject_reason(c["lines"], c["ends"])
        if why:
            c["pass"] = False; c["reasons"].append(why)
        else:
            ok.append(c)
    if not ok:
        return None
    # highest judge score first, then the least used rhyme pairs, which spreads the rhyme vocabulary
    return min(ok, key=lambda c: (-(c["score"] or 0),
                                  max(corpus.pair_uses[p] for p in corpus.pairs(c["ends"]))))

def run_chunk(jobs, corpus):
    t0 = time.time()
    outs = chat([j["messages"] for j in jobs], params, progress=True)
    toks, kept = 0, []
    for j, o in zip(jobs, outs):
        j["cands"] = []
        for c in o.outputs:
            toks += len(c.token_ids)
            plan, lines, tags = parse(c.text)
            r = check(lines)
            r.update(raw=c.text, plan=plan, tags=tags, score=None)
            j["cands"].append(r)
    if JUDGE:
        todo = [(j, c) for j in jobs for c in j["cands"] if c["form_pass"]]
        for (j, c), s in zip(todo, judge([(j["topic"], c["lines"]) for j, c in todo])):
            c["score"] = s
            if s < JUDGE_MIN:
                c["pass"] = False; c["reasons"].append(f"judge={s}")
    for j in jobs:
        best = pick(j["cands"], corpus)
        if best:
            e = best["ends"]
            rec = {"id": j["id"], "topic": j["topic"], "tone": j["tone"], "voice": j["voice"],
                   "lines": best["lines"], "rhymes": [e[0:2], e[2:4]], "syllables": best["syllables"],
                   "score": best["score"], "model": MODEL_ID}
            corpus.add(rec)
            kept.append(rec)
    return kept, time.time() - t0, toks

def new_corpus():
    corpus = Corpus()
    for ex in EXAMPLES:                    # so copies of the worked examples get rejected
        corpus.add({"lines": ex["lines"]}, store=False)
    return corpus
print("checks ok")

checks ok


## 8. Run

Pilot mode runs one batch. Full mode runs chunk after chunk and appends to `gen_out/<model>/quatrains.jsonl`. If it stops for any reason, rerun the whole notebook: it reloads what is saved and continues from the next chunk.

In [8]:
if MODE == "pilot":
    corpus = new_corpus()
    jobs = make_jobs(PILOT_PROMPTS, random.Random(SEED), 0)
    kept, secs, toks = run_chunk(jobs, corpus)
    print(f"\n{len(jobs)} prompts x {N_CANDIDATES} candidates in {secs:.1f}s (incl. judge) | "
          f"{toks:,} output tokens | kept {len(kept)}")
    run_stats = {"prompts": len(jobs), "kept": len(kept), "seconds": secs, "tokens": toks}

else:
    KEPT_FILE, LOG_FILE = f"{OUT_DIR}/quatrains.jsonl", f"{OUT_DIR}/progress.jsonl"
    corpus = new_corpus()
    done = sum(1 for _ in open(LOG_FILE)) if os.path.exists(LOG_FILE) else 0
    if os.path.exists(KEPT_FILE):
        saved = [json.loads(l) for l in open(KEPT_FILE, encoding="utf-8")]
        good = [r for r in saved if r["chunk"] < done]     # drop a half-saved chunk from a crash
        if len(good) != len(saved):
            with open(KEPT_FILE, "w", encoding="utf-8") as f:
                f.writelines(json.dumps(r) + "\n" for r in good)
        for r in good:
            corpus.add(r)
        print(f"Resuming after chunk {done}: {len(corpus.items):,} quatrains already saved")

    run_stats = {"prompts": 0, "kept": 0, "seconds": 0.0, "tokens": 0}
    jobs, chunk = [], done
    while len(corpus.items) < TARGET_USABLE and chunk < MAX_CHUNKS:
        before = len(corpus.items)
        jobs = make_jobs(CHUNK_PROMPTS, random.Random(SEED * 100_003 + chunk), chunk * CHUNK_PROMPTS)
        kept, secs, toks = run_chunk(jobs, corpus)
        kept = kept[:TARGET_USABLE - before]              # don't overshoot the target
        del corpus.items[before + len(kept):]
        for r in kept:
            r["chunk"] = chunk
        with open(KEPT_FILE, "a", encoding="utf-8") as f:
            f.writelines(json.dumps(r) + "\n" for r in kept)
        reasons = Counter(x for j in jobs for c in j["cands"] for x in c["reasons"])
        with open(LOG_FILE, "a") as f:
            f.write(json.dumps({"chunk": chunk, "prompts": len(jobs), "kept": len(kept), "seconds": round(secs, 1),
                                "tokens": toks, "total": len(corpus.items), "reasons": dict(reasons)}) + "\n")
        for k, v in (("prompts", len(jobs)), ("kept", len(kept)), ("seconds", secs), ("tokens", toks)):
            run_stats[k] += v
        print(f"chunk {chunk}: kept {len(kept)}/{len(jobs)} ({len(kept)/len(jobs):.0%}) in {secs:.0f}s | "
              f"total {len(corpus.items):,}/{TARGET_USABLE:,} | top reasons {reasons.most_common(3)}")
        chunk += 1
        if len(kept) < 0.05 * len(jobs) and len(corpus.items) < TARGET_USABLE:
            print("Yield fell below 5% of prompts, stopping. Check the reasons above: usually pair_cap "
                  "(raise MAX_PAIR_USES or add topics) or near_duplicate (raise TEMPERATURE).")
            break
    print(f"\nDone: {len(corpus.items):,} quatrains in {KEPT_FILE}")

Processed prompts: 100%|██████████| 1200/1200 [01:08<00:00, 17.44it/s, est. speed input: 13512.12 toks/s, output: 1207.33 toks/s] 



300 prompts x 4 candidates in 82.6s (incl. judge) | 83,094 output tokens | kept 128


## 9. Stats

In full mode, the candidate-level numbers are for the last chunk and the dataset-level numbers cover everything saved.

In [9]:
import pandas as pd

summary = {}
cands = [c for j in jobs for c in j["cands"]]
if cands:
    form_ok = sum(c["form_pass"] for c in cands)
    kept_prompts = sum(any(c["pass"] for c in j["cands"]) for j in jobs)
    four = [c for c in cands if len(c["lines"]) == 4 and len(c["tags"]) == 4]
    tag_hit = sum(t == last_word(l) for c in four for t, l in zip(c["tags"], c["lines"])) / max(4 * len(four), 1)
    summary.update(form_pass=form_ok / len(cands), tag_adherence=tag_hit, yield_=kept_prompts / len(jobs))
    print(f"Candidates passing form checks: {form_ok}/{len(cands)} ({form_ok/len(cands):.0%})")
    print(f"Lines ending with their [tag] word: {tag_hit:.0%}   (v2 plan adherence was ~40%)")
    print(f"Prompts with a kept quatrain:   {kept_prompts}/{len(jobs)} ({kept_prompts/len(jobs):.0%})\n")

    scores = [c["score"] for c in cands if c["score"] is not None]
    if scores:
        dist = Counter(scores)
        summary.update(judge_pass=sum(s >= JUDGE_MIN for s in scores) / len(scores),
                       judge_mean=sum(scores) / len(scores))
        print(f"Judge scores for form-passing candidates (keep >= {JUDGE_MIN}):",
              {s: dist[s] for s in range(0, 6) if dist[s]},
              f"| mean {summary['judge_mean']:.2f} | pass {summary['judge_pass']:.0%}\n")

    print("Failure reasons per candidate (one can fail for several):")
    for r, n in Counter(x for c in cands for x in c["reasons"]).most_common():
        print(f"  {r:16s} {n:6d}  ({n/len(cands):.0%})")

items = corpus.items
syl = [s for r in items for s in r["syllables"]]
if syl:
    print(f"\nSyllables per line (kept): mean {sum(syl)/len(syl):.1f}, min {min(syl)}, max {max(syl)}")
    print("  distribution:", dict(sorted(Counter(syl).items())))

pairs = Counter(tuple(p) for r in items for p in r["rhymes"])
print(f"\nDataset: {len(items):,} quatrains | {len(pairs):,} distinct rhyme pairs")
print("Most used pairs:", pairs.most_common(10))
sounds = Counter(min(rhyme_parts(p[0]) or {"?"}) for r in items for p in r["rhymes"])
print(f"Distinct rhyme sounds: {len(sounds)}; most common:", sounds.most_common(6))

ws = [w for r in items for l in r["lines"] for w in words_of(l)]
if ws:
    print(f"Vocabulary: {len(set(ws)):,} distinct words in {len(ws):,} tokens (type/token {len(set(ws))/len(ws):.2f})")
print("Kept by tone:", dict(Counter(r["tone"] for r in items).most_common()))

Candidates passing form checks: 480/1200 (40%)
Lines ending with their [tag] word: 77%   (v2 plan adherence was ~40%)
Prompts with a kept quatrain:   128/300 (43%)

Judge scores for form-passing candidates (keep >= 4): {2: 2, 3: 304, 4: 173, 5: 1} | mean 3.36 | pass 36%

Failure reasons per candidate (one can fail for several):
  B_no_rhyme          342  (28%)
  A_no_rhyme          321  (27%)
  judge=3             304  (25%)
  syllable_range      291  (24%)
  rare_end_word        55  (5%)
  pair_unbalanced      40  (3%)
  line_count=2          3  (0%)
  same_sound_AB         3  (0%)
  judge=2               2  (0%)
  line_count=8          1  (0%)
  line_count=3          1  (0%)
  line_count=6          1  (0%)

Syllables per line (kept): mean 10.2, min 7, max 12
  distribution: {7: 2, 8: 18, 9: 88, 10: 201, 11: 153, 12: 50}

Dataset: 128 quatrains | 227 distinct rhyme pairs
Most used pairs: [(('gone', 'dawn'), 5), (('hope', 'cope'), 4), (('bright', 'light'), 3), (('find', 'mind'), 3), ((

## 10. Samples

Read the **judge-rejected** group as well as the kept one: it shows whether the judge is too strict or too lenient. If good poems are being rejected, set `JUDGE_MIN = 3`.

In [10]:
def show(c, head=""):
    print(head)
    for l, s in zip(c["lines"], c.get("syllables") or [None] * 8):
        print(f"   {l}" + (f"   ({s})" if s else ""))
    if c.get("reasons"):
        print("   FAIL:", ", ".join(c["reasons"]))
    print()

rng = random.Random()
batch_ids = {j["id"] for j in jobs}
kept_now = [r for r in corpus.items if r["id"] in batch_ids]
print("=========== KEPT (random 12) ===========\n")
for r in rng.sample(kept_now, min(12, len(kept_now))):
    show(r, f"[{r['id']}] {r['topic']} | {r['tone']} | {r['voice']} | score {r['score']} | rhymes {r['rhymes']}")

judged_out = [(j, c) for j in jobs for c in j["cands"] if c["form_pass"] and c["score"] is not None and c["score"] < JUDGE_MIN]
print(f"=========== PASSED FORM, REJECTED BY JUDGE (random 8 of {len(judged_out)}) ===========\n")
for j, c in rng.sample(judged_out, min(8, len(judged_out))):
    show(c, f"[{j['id']}] {j['topic']} | {j['tone']} | score {c['score']}")

failed = [(j, c) for j in jobs for c in j["cands"] if not c["form_pass"]]
print(f"=========== FAILED FORM CHECKS (random 6 of {len(failed)}) ===========\n")
for j, c in rng.sample(failed, min(6, len(failed))):
    show(c, f"[{j['id']}] {j['topic']} | {j['tone']} | plan: {c['plan']} | tags: {c['tags']}")

=========== KEPT (random 12) ===========

[183] scrolling past bedtime | bittersweet | a narrator describing someone | score 4 | rhymes [['late', 'state'], ['slow', 'show']]
   She scrolls past midnight, her screen still late,   (9)
   lost in the endless social media state.   (11)
   The backlight fades as the time runs slow,   (9)
   she closes her eyes, the digital show.   (10)

[57] coffee mug left on counter | bittersweet | a narrator describing someone | score 4 | rhymes [['gone', 'dawn'], ['keep', 'deep']]
   She left her coffee mug there when she went gone,   (11)
   now steam rises with the morning's first dawn.   (10)
   In its warmth, he finds a trace he wants to keep,   (11)
   a memory etched as dark as the brew deep.   (11)

[5] a beggar holding a cup | cheerful | a narrator describing someone | score 4 | rhymes [['sky', 'high'], ['dreams', 'themes']]
   He looks up to the clouds in the sky,   (9)
   a smile as broad as a mountain high.   (9)
   His eyes hold stories of d

## 11. Estimate and save

In [11]:
secs = max(run_stats["seconds"], 1e-6)
yield_per_prompt = run_stats["kept"] / max(run_stats["prompts"], 1)
prompts_per_sec = run_stats["prompts"] / secs
print(f"Yield: {yield_per_prompt:.0%} of prompts give a kept quatrain | {prompts_per_sec:.1f} prompts/s "
      f"| {run_stats['tokens']/secs:,.0f} generation tok/s")
est = {}
if yield_per_prompt > 0:
    for target in (20_000, 50_000):
        n = target / yield_per_prompt
        est[target] = n / prompts_per_sec / 60
        print(f"  {target:,} quatrains -> about {n:,.0f} prompts -> roughly {est[target]:.0f} min")
    print("Larger chunks run faster per prompt, but yield drops a little as rhyme-pair caps fill, so treat this as a rough guide.")

def dump(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        f.writelines(json.dumps(r) + "\n" for r in rows)

if MODE == "pilot":
    dump(f"{OUT_DIR}/pilot_kept.jsonl", corpus.items)
    dump(f"{OUT_DIR}/pilot_all.jsonl", [{k: v for k, v in j.items() if k != "messages"} for j in jobs])
kept_scores = [r["score"] for r in corpus.items if r.get("score")]
meta = {"mode": MODE, "model_key": MODEL, "model": MODEL_ID, "n_candidates": N_CANDIDATES,
        "temperature": TEMPERATURE, "top_p": TOP_P, "judge_min": JUDGE_MIN if JUDGE else None,
        "saved": len(corpus.items), **run_stats, **summary,
        "kept_score_mean": sum(kept_scores) / len(kept_scores) if kept_scores else None,
        "est_min_20k": est.get(20_000), "est_min_50k": est.get(50_000)}
json.dump(meta, open(f"{OUT_DIR}/meta.json", "w"), indent=2)
print("Saved to", OUT_DIR)

Yield: 43% of prompts give a kept quatrain | 3.6 prompts/s | 1,006 generation tok/s
  20,000 quatrains -> about 46,875 prompts -> roughly 215 min
  50,000 quatrains -> about 117,187 prompts -> roughly 538 min
Larger chunks run faster per prompt, but yield drops a little as rhyme-pair caps fill, so treat this as a rough guide.
Saved to pilot_v3_out/qwen2.5-32b


## 12. Compare models

Reads every pilot in `pilot_v3_out/`. Run the pilot once per model (change `MODEL`, restart the kernel, Run All), then look here. Pick by **yield × reading quality**, not speed alone: the judge is the model rating itself, so also compare the kept samples by eye.

In [12]:
rows = []
for key in sorted(os.listdir(PILOT_ROOT)) if os.path.isdir(PILOT_ROOT) else []:
    p = f"{PILOT_ROOT}/{key}/meta.json"
    if os.path.exists(p):
        m = json.load(open(p))
        rows.append({
            "model": key,
            "form pass": f"{m.get('form_pass', 0):.0%}",
            "tag adherence": f"{m.get('tag_adherence', 0):.0%}",
            "judge mean": round(m["judge_mean"], 2) if m.get("judge_mean") else None,
            "judge pass": f"{m.get('judge_pass', 0):.0%}",
            "yield/prompt": f"{m.get('yield_', 0):.0%}",
            "kept": m["kept"],
            "pilot secs": round(m["seconds"]),
            "est min 20k": round(m["est_min_20k"]) if m.get("est_min_20k") else None,
        })
print(pd.DataFrame(rows).to_string(index=False) if rows else "No pilots yet.")

        model form pass tag adherence  judge mean judge pass yield/prompt  kept  pilot secs  est min 20k
  qwen2.5-32b       40%           77%        3.36        36%          43%   128          83          215
qwen3-30b-a3b       14%           49%        4.01        74%          35%   106          25           78
    qwen3-32b       30%           79%        3.85        65%          56%   169         134          265


## What to check before switching to full

- **Tag adherence** should be well above v2's ~40%. If it is still low for every model, the model is not holding the end word, and fixed end words would need constrained decoding.
- **Judge-rejected samples:** are they really worse than the kept ones? If the judge rejects good poems, lower `JUDGE_MIN` to 3. If kept poems still have nonsense phrases, the judge is too lenient for that model; prefer a model whose kept samples read better.
- **Copies of the examples:** check that kept quatrains are not reworded versions of the four worked examples.
- **Most used rhyme pairs:** if a few pairs dominate, the cap will start rejecting a lot in the full run.
- When happy: set `MODE = "full"` and the chosen `MODEL`, then Restart & Run All.

Output format (`gen_out/<model>/quatrains.jsonl`), one per line:
`{"id", "topic", "tone", "voice", "lines": [4 strings], "rhymes": [["late","wait"],["phone","alone"]], "syllables", "score", "model", "chunk"}`.
`rhymes` is the conditioning label for the generator, read from the quatrain itself.